# 🎾 OMNIS-COURT LLM + Jina Server
## Colab Primary Instance

**Steps:**
1. Runtime → Change runtime type → T4 GPU
2. Run All
3. Wait ~10 min for model download
4. Copy both URLs when printed
5. Paste into config/platforms.json
6. Close tab safely (anti-idle active)

In [ ]:
# ==========================================
# CELL 1: INSTALL DEPENDENCIES
# ==========================================
!pip install -q vllm trafilatura fastapi uvicorn nest-asyncio
!curl -sL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

import subprocess
cf = subprocess.run(['cloudflared', '--version'], capture_output=True, text=True)
print(f'✅ cloudflared: {cf.stdout.strip()}')
for pkg in ['vllm', 'trafilatura', 'fastapi', 'uvicorn']:
    try:
        __import__(pkg.replace('-','_'))
        print(f'✅ {pkg}')
    except:
        print(f'❌ {pkg}')
print('✅ Ready for Cell 2')

In [ ]:
# ==========================================
# CELL 2: ANTI-IDLE PROTECTION
# ==========================================
from IPython.display import display, Javascript

display(Javascript('''
    setInterval(function(){
        var b=document.querySelector("colab-run-button");
        if(b)b.click();
    },300000);
'''))
print('✅ Anti-idle active! Safe to close tab after all cells run.')

In [ ]:
# ==========================================
# CELL 3: START QWEN3 VLLM SERVER
# ==========================================
import subprocess, time, requests

proc = subprocess.Popen([
    'python','-m','vllm.entrypoints.openai.api_server',
    '--model','Qwen/Qwen3-30B-A3B',
    '--served-model-name','qwen3-30b',
    '--host','0.0.0.0','--port','8000',
    '--max-model-len','8192',
    '--gpu-memory-utilization','0.9',
    '--trust-remote-code','--enforce-eager'
], stdout=subprocess.PIPE, stderr=subprocess.PIPE)

print('🚀 Starting vLLM... (~10 min first run)')
for i in range(60):
    try:
        r = requests.get('http://localhost:8000/health', timeout=2)
        if r.status_code == 200:
            print('✅ vLLM READY on port 8000')
            break
    except:
        pass
    time.sleep(10)
    if i % 6 == 0:
        print(f'⏳ Waiting... {(i+1)*10}s')
else:
    print('❌ Failed. Last logs:')
    print(proc.stderr.read().decode()[-1000:])

In [ ]:
# ==========================================
# CELL 4: START JINA READER SERVER
# ==========================================
import threading, time, requests as req
import trafilatura
from fastapi import FastAPI, Query
from fastapi.responses import JSONResponse
import uvicorn, nest_asyncio
nest_asyncio.apply()

app = FastAPI()

@app.get('/health')
async def health():
    return {'status':'ok'}

@app.get('/extract')
async def extract(url: str = Query(...)):
    try:
        dl = trafilatura.fetch_url(url)
        if not dl:
            return JSONResponse(400, content={'error':'fetch failed','url':url})
        txt = trafilatura.extract(dl, include_comments=False, include_tables=True, no_fallback=False)
        if not txt or len(txt.strip()) < 50:
            return JSONResponse(400, content={'error':'content too short','url':url})
        return {'url':url,'content':txt,'word_count':len(txt.split()),'status':'success'}
    except Exception as e:
        return JSONResponse(500, content={'error':str(e),'url':url})

def run():
    uvicorn.run(app, host='0.0.0.0', port=8001, log_level='warning')

t = threading.Thread(target=run, daemon=True)
t.start()
time.sleep(3)
try:
    r = req.get('http://localhost:8001/health', timeout=5)
    print('✅ Jina Reader READY on port 8001' if r.status_code==200 else '❌ Jina error')
except Exception as e:
    print(f'❌ Jina failed: {e}')

In [ ]:
# ==========================================
# CELL 5: CLOUDFLARE TUNNELS
# ==========================================
import subprocess, re

def tunnel(port):
    p = subprocess.Popen(
        ['cloudflared','tunnel','--url',f'http://localhost:{port}'],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
    )
    for line in p.stderr:
        m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
        if m:
            return p, m.group(0)
    return p, None

print('🌐 Tunnel LLM (8000)...')
p1, u1 = tunnel(8000)
print('🌐 Tunnel Jina (8001)...')
p2, u2 = tunnel(8001)

if u1 and u2:
    print('\n' + '='*60)
    print('🎉 OMNIS-COURT COLAB READY!')
    print('='*60)
    print(f'🧠 LLM:  {u1}')
    print(f'📖 JINA: {u2}')
    print('='*60)
    print('📋 COPY BOTH URLs → config/platforms.json')
    print('🔒 Anti-idle ON → safe to close tab')
else:
    print(f'❌ Tunnel failed: LLM={u1}, Jina={u2}')

In [ ]:
# ==========================================
# CELL 6: TEST BOTH SERVICES
# ==========================================
import requests

print('🧪 Testing LLM...')
try:
    r = requests.post(
        f'{u1}/v1/chat/completions',
        json={'model':'qwen3-30b','messages':[{'role':'user','content':'Say OK'}],'max_tokens':5},
        timeout=30
    )
    if r.status_code == 200:
        print(f"✅ LLM: {r.json()['choices'][0]['message']['content']}")
    else:
        print(f'❌ LLM: {r.status_code}')
except Exception as e:
    print(f'❌ LLM: {e}')

print('🧪 Testing Jina...')
try:
    r = requests.get(
        f'{u2}/extract',
        params={'url':'https://en.wikipedia.org/wiki/Tennis'},
        timeout=30
    )
    if r.status_code == 200:
        print(f"✅ Jina: {r.json()['word_count']} words")
    else:
        print(f'❌ Jina: {r.status_code}')
except Exception as e:
    print(f'❌ Jina: {e}')

print('\n✅ All tests passed!' if u1 and u2 else '\n❌ Some tests failed')